In [ ]:
!pip install -q chromadb

In [1]:
import json
import numpy as np

documents_path = "../data/documents.json"
embeddings_path = "../data/hadith_embeddings.npy"

with open(documents_path, "r", encoding="utf-8") as f:
    documents = json.load(f)

embeddings = np.load(embeddings_path)

print("Documents:", len(documents))
print("Embeddings:", embeddings.shape)

Documents: 116
Embeddings: (116, 768)


#  Model Downloading


In [2]:
import os
import re

from dotenv import find_dotenv, load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer

# توكن HuggingFace من .env فقط
load_dotenv(find_dotenv())
login(token=os.environ["huggingface_Access_Tokens"])

# توحيد النص قبل الترميز (نفس ما استخدم في بناء الإمبدنجز)
TASHKEEL = "".join(chr(c) for c in list(range(0x064B, 0x0660)) + [0x0670])


def normalize_ar(text):
    text = text.translate(str.maketrans("", "", TASHKEEL))
    return re.sub(r"\s+", " ", text).strip()


MODEL_NAME = "omarelshehy/Arabic-Retrieval-v1.0"

embedding_model = SentenceTransformer(MODEL_NAME, device="cpu")

print("Model loaded successfully!")
print("Embedding dimension:", embedding_model.get_embedding_dimension())


Model loaded successfully!
Embedding dimension: 768


In [3]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="../chroma_db"
)

# الأبعاد تغيرت (384 -> 768) فنبدأ مجموعة نظيفة
try:
    chroma_client.delete_collection(name="arbaeen_nawawi_small")
    print("Old collection deleted.")
except Exception:
    print("No old collection found.")

collection = chroma_client.create_collection(
    name="arbaeen_nawawi_small"
)

print("Collection created successfully!")
print("Collection name:", collection.name)
print("Current documents:", collection.count())


Old collection deleted.
Collection created successfully!
Collection name: arbaeen_nawawi_small
Current documents: 0


In [4]:
assert embeddings.shape[1] == embedding_model.get_embedding_dimension(), "dim mismatch: rebuild embeddings"
assert embeddings.shape[0] == len(documents), "docs/embeddings count mismatch"

ids = []
texts = []
metadatas = []
embeddings_list = []

for i, doc in enumerate(documents):
    ids.append(f"doc_{i}")
    texts.append(doc["content"])

    metadata = doc["metadata"].copy()

    # ChromaDB لا تقبل list داخل metadata
    for key, value in metadata.items():
        if isinstance(value, list):
            metadata[key] = ", ".join(map(str, value))

    metadatas.append(metadata)
    embeddings_list.append(embeddings[i].tolist())

print("Prepared documents:", len(ids))
print("Prepared embeddings:", len(embeddings_list))

Prepared documents: 116
Prepared embeddings: 116


In [5]:
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings_list,
    metadatas=metadatas
)

print("Documents added successfully!")
print("Total documents in collection:", collection.count())

Documents added successfully!
Total documents in collection: 116


In [6]:
result = collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

for i in range(3):
    print("=" * 60)
    print("ID:", result["ids"][i])
    print("Metadata:", result["metadatas"][i])
    print("Content:", result["documents"][i][:300])

ID: doc_0
Metadata: {'narrator': 'أمير المؤمنين أبي حفص عمر بن الخطاب ﴿رضي الله تعالى عنه﴾', 'type': 'hadith', 'pages': '3', 'title': 'الأعمال بالنيات', 'hadith_number': 1}
Content: الأعمال بالنيات

عن أمير المؤمنين أبي حفص عمر بن الخطاب قال: سمعت رسول الله ﷺ يقول: " إنما الأعمال بالنيات، وإنما لكل امرىء ما نوى، فمن كانت هجرته إلى الله ورسوله فهجرته إلى الله ورسوله، ومن كانت هجرته لدنيا يصيبها، أو امرأة ينكحها، فهجرته إلى ما هاجر إليه " رواه إماما المحدثين أبو عبدالله محمد بن إ
ID: doc_1
Metadata: {'hadith_number': 2, 'type': 'hadith', 'narrator': 'عمر ﴿رضي الله تعالى عنه﴾', 'pages': '4, 5', 'title': 'مراتب الدين'}
Content: مراتب الدين

عن عمر رضي الله تعالى عنه أقال: بينما نحن جلوس عند رسول الله ذات يوم إذ طلع علينا رجل شديد بياض الثياب شديد سواد الشعر لا يرى عليه أثر السفر ولا يعرفه منا أحد حتى جلس إلى النبي فأسند ركبتيه إلى ركبتيه ووضع كفيه على فخذيه وقال: يا محمد أخبرني عن الإسلام، فقال رسول الله ﷺ: " الإسلام أن تش
ID: doc_2
Metadata: {'type': 'hadith', 'hadith_number': 3, 'title':

In [7]:
query = "ما الحديث الذي يتحدث عن حسن الخلق "

query_embedding = embedding_model.encode_query(
    normalize_ar(query),
    normalize_embeddings=True
)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for i in range(3):
    print("=" * 60)
    print(f"Result {i + 1}")
    print("Distance:", results["distances"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print("Content:", results["documents"][0][i][:500])


Result 1
Distance: 0.6119334101676941
Metadata: {'type': 'hadith', 'narrator': 'أبي ذر جندب بن جنادة وأبي عبد الرحمن معاذ بن جبل ﴿رضي الله تعالى عنه﴾', 'hadith_number': 18, 'pages': '13, 14', 'title': 'الخلق الحسن'}
Content: الخلق الحسن

عن أبي ذر جندب بن جنادة وأبي عبد الرحمن معاذ بن جبل رضي الله عنهما عن رسول الله ﷺ قال: (اتق الله حيثما كنت، وأتبع السيئة الحسنة تمحها، وخالق الناس بخلق حسن ) رواه الترمذي وقال: حديث حسن. وفي بعض النسخ: حسن صحيح.
Result 2
Distance: 0.742151141166687
Metadata: {'pages': '13, 14', 'type': 'sharh', 'hadith_number': 18, 'title': 'الخلق الحسن'}
Content: الخلق الحسن

قوله: (اتق الله) أي اتخذ وقاية من عذاب الله عز وجل، وذلك بفعل أوامره واجتناب نواهيه. (حيثما كنت) حيث: ظرف مكان، أي في أي مكان كنت سواء في العلانية أو في السر، وسواء في البيت أو في السوق، وسواء عندك أناس أو ليس عندك أناس. ( وأتبع السيئة الحسنة تمحها) (أتبع) فعل أمر، و (السيئة) مفعول أول، و (الحسنة) مفعول ثان. ( تمحها) جواب الأمر، ولهذا جزمت، والمعنى: إذا فعلت سيئة فأتبعها بحسنة، فهذه الحسنة تمحو ا